# NB03 — Standard RAG Pipeline (Method 3)
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: End-to-end standard RAG pipeline from ingest through evaluation.  
All implementation code is drawn directly from `src/methods/standard_rag/langchain_pipeline.py`.

**Stack**: LangChain 0.3.x · langchain-chroma · DeepEval (Confident AI) · GPT-4o

**Pipeline stages**
| # | Stage | Key function |
|---|-------|-------------|
| 1 | Ingest | `load_from_pubmed`, `load_from_pdfs`, `load_from_json` |
| 2 | Chunk | `chunk_documents` (section-aware `RecursiveCharacterTextSplitter`) |
| 3+4 | Embed + Store | `build_vector_store` / `load_vector_store` (Chroma, persistent) |
| 5 | RAG chain (LCEL) | `build_rag_chain` → `RunnablePassthrough` + `StrOutputParser` |
| 6 | Evaluate | `evaluate_pipeline` → 5 DeepEval RAG metrics (LLM-as-judge) |
| 7 | NIAH | `run_niah_test` → depth × context-length sweep heatmap |

**Outputs**
- `data/processed/standard_rag_results.csv`
- `reports/standard_rag_deepeval_scores.csv`
- `reports/niah_results_standard_rag.json`
- `experiments/<run_id>.json`

In [ ]:
import sys
sys.path.insert(0, '..')

# ── env ──────────────────────────────────────────────────────────────────────
import os
import json
import time
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
load_dotenv()

# ── LangChain ────────────────────────────────────────────────────────────────
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFDirectoryLoader, PubMedLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── DeepEval ─────────────────────────────────────────────────────────────────
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
)

# ── Project utils ─────────────────────────────────────────────────────────────
from src.utils.io import data_dir, project_root, generate_run_id, log_experiment

import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
CHROMA_DIR      = str(project_root() / "chroma_db")
COLLECTION      = "systematic_review"
EMBED_MODEL     = "text-embedding-3-small"   # 1536-dim, $0.02/1M tokens
LLM_MODEL       = "gpt-4o"
TOP_K           = 8
CHUNK_SIZE      = 800
CHUNK_OVERLAP   = 150
EVAL_MODEL      = "gpt-4o"

# NIAH sweep grid (matches NB08 for direct comparability)
NIAH_CONTEXT_LENGTHS = [2_000, 8_000, 16_000, 32_000]
NIAH_DEPTHS          = [0.1, 0.25, 0.5, 0.75, 0.9]

print("Imports OK. CHROMA_DIR:", CHROMA_DIR)

## Stage 1 — Ingest

Three supported sources — use whichever matches your corpus:

| Loader | Use case |
|--------|----------|
| `PubMedLoader` | PubMed abstracts via the same Boolean query as Method 1 |
| `PyPDFDirectoryLoader` | Full-text PDFs from Embase / WoS / Cochrane |
| JSON | Covidence / Rayyan exports |

Run **one** of the cells below, then proceed to Stage 2.

In [ ]:
## 1a. PubMed abstracts  ← use the same Boolean query as Method 1 for alignment

PUBMED_QUERY = (
    '("large language model"[tiab] OR "LLM"[tiab] OR "GPT"[tiab] OR '
    '"foundation model"[tiab]) '
    'AND ("feature extraction"[tiab] OR "information extraction"[tiab] OR '
    '"named entity"[tiab]) '
    'AND ("radiology report"[tiab] OR "pathology report"[tiab] OR '
    '"clinical note"[tiab]) '
    'AND ("2018/01/01"[PDAT] : "2026/12/31"[PDAT])'
)

loader = PubMedLoader(query=PUBMED_QUERY, load_max_docs=500)
docs = loader.load()
print(f"[ingest:pubmed] {len(docs)} abstracts loaded")

# Preview first record
print("\n── Sample document ──")
print("Content:", docs[0].page_content[:300])
print("Metadata:", docs[0].metadata)

## 1b. Full-text PDFs  ← skip if using PubMed abstracts only

In [ ]:
## 1b — PDF ingest (uncomment to use)
# PDF_DIR = os.environ.get("RAW_PDF_DIR", "../data_private/raw")
# loader = PyPDFDirectoryLoader(PDF_DIR, glob="**/*.pdf")
# docs = loader.load()
# print(f"[ingest:pdf] {len(docs)} pages from {PDF_DIR}")

## 1c — JSON (Covidence / Rayyan export)
# def load_from_json(json_path: str) -> list[Document]:
#     records = json.loads(Path(json_path).read_text())
#     return [
#         Document(
#             page_content=f"Title: {r.get('title','')}\n\nAbstract: {r.get('abstract','')}",
#             metadata={"title": r.get("title",""), "authors": r.get("authors",""),
#                       "year": str(r.get("year","")), "source": "json_export"},
#         )
#         for r in records
#     ]
# docs = load_from_json("../data/covidence_export.json")
# print(f"[ingest:json] {len(docs)} records loaded")

print(f"Total documents to process: {len(docs)}")

## Stage 2 — Chunk

Section-aware `RecursiveCharacterTextSplitter`: tries biomedical section headers (Abstract → Introduction → Methods → Results → Discussion → Conclusion) before falling back to paragraph → sentence → word boundaries.

- `chunk_size = 800` chars (~200 tokens) — fits comfortably within GPT-4o context alongside other chunks
- `chunk_overlap = 150` chars — preserves cross-boundary evidence sentences
- `add_start_index = True` → each chunk carries its byte offset for provenance

In [ ]:
_SECTION_SEPS = [
    "\nAbstract\n", "\nINTRODUCTION\n", "\nIntroduction\n",
    "\nMETHODS\n",  "\nMethods\n",       "\nMaterials and Methods\n",
    "\nRESULTS\n",  "\nResults\n",
    "\nDISCUSSION\n", "\nDiscussion\n",
    "\nCONCLUSION\n", "\nConclusion\n",
    "\nREFERENCES\n", "\n\n", "\n", ". ", " ",
]

splitter = RecursiveCharacterTextSplitter(
    separators=_SECTION_SEPS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    add_start_index=True,
)

chunks = splitter.split_documents(docs)

# Tag each chunk with its index within its source document (for provenance)
counters: dict[str, int] = {}
for c in chunks:
    src = c.metadata.get("source", "unknown")
    counters[src] = counters.get(src, 0) + 1
    c.metadata["chunk_index"] = counters[src]

print(f"[chunk] {len(docs)} docs → {len(chunks)} chunks")
print(f"  mean chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")

# Preview a mid-document chunk
sample = chunks[len(chunks) // 2]
print("\n── Sample chunk ──")
print(f"Source: {sample.metadata.get('title', sample.metadata.get('source'))}")
print(f"Chunk index: {sample.metadata.get('chunk_index')}  |  start_index: {sample.metadata.get('start_index')}")
print("Content:", sample.page_content[:300])

## Stage 3 + 4 — Embed & Store (Chroma)

`text-embedding-3-small` (1 536-dim) is used for cost efficiency across a potentially large corpus.  
Switch to `text-embedding-3-large` (3 072-dim) in `EMBED_MODEL` above for higher accuracy runs.

> **Run once** — after the first run comment out `build_vector_store` and use `load_vector_store` instead.

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL, openai_api_key=OPENAI_API_KEY)

# ── First run: build and persist ──────────────────────────────────────────────
vs = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION,
    persist_directory=CHROMA_DIR,
)
print(f"[store] {vs._collection.count()} vectors persisted → {CHROMA_DIR}")

# ── Subsequent runs: load from disk (no re-embedding) ─────────────────────────
# vs = Chroma(
#     collection_name=COLLECTION,
#     embedding_function=embeddings,
#     persist_directory=CHROMA_DIR,
# )
# print(f"[store] Loaded {vs._collection.count()} vectors from {CHROMA_DIR}")

## Stage 5 — RAG Chain (LCEL)

Built with **LangChain Expression Language (LCEL)**:

```
retriever → format_docs ─┐
                          ├→ prompt → gpt-4o → StrOutputParser
passthrough ──────────────┘
```

`_format_docs` injects `[Title (Year)]` citations into the context string so the LLM can attribute claims to specific sources in its answer.

In [ ]:
_REVIEW_PROMPT = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a systematic review researcher.\n"
        "Answer using ONLY the retrieved study excerpts in <context>.\n"
        "For each claim, cite the source title and year in parentheses.\n"
        "If the evidence is insufficient, say so — do not speculate.\n\n"
        "Structure your answer as:\n"
        "1. Summary of evidence (2–3 sentences)\n"
        "2. Key findings (bullet list, cited)\n"
        "3. Gaps and limitations\n"
        "4. PICO elements for formal meta-analysis\n\n"
        "<context>\n{context}\n</context>"
    )),
    ("human", "{question}"),
])

def _format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[{doc.metadata.get('title', doc.metadata.get('source', 'unknown'))} "
        f"({doc.metadata.get('year', 'n.d.')})] {doc.page_content}"
        for doc in docs
    )

retriever = vs.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})
llm = ChatOpenAI(model=LLM_MODEL, temperature=0, openai_api_key=OPENAI_API_KEY)

chain = (
    {"context": retriever | _format_docs, "question": RunnablePassthrough()}
    | _REVIEW_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")
print(f"  Retriever: similarity, k={TOP_K}")
print(f"  LLM: {LLM_MODEL}, temperature=0")

In [ ]:
## Quick retrieval test — inspect answer + source documents

def run_query(question: str) -> dict[str, Any]:
    """Run a question through the chain; return answer, context, and source metadata."""
    answer = chain.invoke(question)
    retrieved_docs = retriever.invoke(question)
    retrieval_context = [d.page_content for d in retrieved_docs]
    sources = [
        {
            "title":   d.metadata.get("title", d.metadata.get("source", "?")),
            "year":    d.metadata.get("year", ""),
            "chunk":   d.metadata.get("chunk_index", ""),
            "excerpt": d.page_content[:200] + "…",
        }
        for d in retrieved_docs
    ]
    return {"question": question, "answer": answer,
            "retrieval_context": retrieval_context, "sources": sources}

test_q = (
    "What LLM optimization strategies have been applied to "
    "radiology report feature extraction?"
)
result = run_query(test_q)

print("=" * 70)
print("ANSWER:\n")
print(result["answer"])
print("\n" + "=" * 70)
print(f"\nSOURCE DOCUMENTS  (top {TOP_K} retrieved)\n")
for i, s in enumerate(result["sources"], 1):
    print(f"[{i}] {s['title']} ({s['year']})  chunk={s['chunk']}")
    print(f"    {s['excerpt']}\n")

## Stage 6 — DeepEval Evaluation (5 RAG metrics, LLM-as-judge)

| Metric | What it tests |
|--------|--------------|
| `FaithfulnessMetric` | Are all claims in the answer supported by the retrieval context? |
| `AnswerRelevancyMetric` | Does the answer address the question (not just fill space)? |
| `ContextualPrecisionMetric` | Are relevant chunks ranked **above** irrelevant ones? |
| `ContextualRecallMetric` | Does the retrieval context cover the expected answer? |
| `ContextualRelevancyMetric` | Is the retrieval context relevant to the question overall? |

Judge model: `gpt-4o` · Threshold: `0.7` · `include_reason=True` for interpretability.  
Ref: [deepeval.com/docs/getting-started-rag](https://deepeval.com/docs/getting-started-rag)

In [ ]:
def make_metrics(threshold: float = 0.7) -> list:
    kwargs = dict(model=EVAL_MODEL, threshold=threshold, include_reason=True)
    return [
        FaithfulnessMetric(**kwargs),
        AnswerRelevancyMetric(**kwargs),
        ContextualPrecisionMetric(**kwargs),
        ContextualRecallMetric(**kwargs),
        ContextualRelevancyMetric(**kwargs),
    ]


# QA pairs — supplement with expected outputs from gold-standard set when available
qa_pairs = [
    {
        "question": "What LLM optimization strategies have been applied to radiology report feature extraction?",
        "expected": None,   # fill in from data/splits/eval_query_set.csv expected column
    },
    {
        "question": "What prompt engineering techniques improve structured data extraction from pathology reports?",
        "expected": None,
    },
    {
        "question": "How does fine-tuning compare to in-context learning for clinical NLP on radiology text?",
        "expected": None,
    },
]

# Optionally load the full eval query set
query_df = pd.read_csv(data_dir() / "splits" / "eval_query_set.csv")
qa_pairs_full = query_df[["question", "expected"]].to_dict("records")

metrics  = make_metrics(threshold=0.7)
test_cases: list[LLMTestCase] = []

print(f"[eval] Building {len(qa_pairs_full)} test cases …")
t0 = time.perf_counter()

for pair in qa_pairs_full:
    res = run_query(pair["question"])
    tc  = LLMTestCase(
        input=pair["question"],
        actual_output=res["answer"],
        retrieval_context=res["retrieval_context"],
        expected_output=pair.get("expected") or res["answer"],
    )
    test_cases.append(tc)

eval_result = evaluate(test_cases=test_cases, metrics=metrics)
elapsed_eval = round(time.perf_counter() - t0, 1)

# Aggregate mean score per metric
agg: dict[str, list[float]] = {}
for tc in test_cases:
    for m in metrics:
        name = type(m).__name__
        agg.setdefault(name, []).append(m.score or 0.0)

summary = {name: round(sum(v) / len(v), 4) for name, v in agg.items()}

print(f"\n[eval] Completed in {elapsed_eval}s\n")
print(f"{'Metric':<30} {'Score':>6}  {'Pass (≥0.7)':>10}")
print("─" * 52)
for name, score in summary.items():
    status = "✓" if score >= 0.7 else "✗"
    print(f"{name:<30} {score:>6.3f}  {status:>10}")

scores_df = pd.DataFrame([summary])
scores_df.to_csv(project_root() / "reports" / "standard_rag_deepeval_scores.csv", index=False)
print("\nScores saved → reports/standard_rag_deepeval_scores.csv")

## Stage 7 — Needle-in-a-Haystack (NIAH) Sweep

Adapted from [Kamradt (2023)](https://github.com/gkamradt/LLMTest_NeedleInAHaystack).

**Protocol per cell**:
1. Build a haystack of `context_length × 4` characters (biomedical filler prose)
2. Insert the needle at `depth` fraction through the document
3. Chunk the haystack with the same splitter used in Stage 2
4. **Temporarily** add chunks to ChromaDB
5. Query the chain — record `needle_retrieved` (retriever), `needle_in_answer` (generator), DeepEval Faithfulness
6. **Delete** temporary chunks from ChromaDB

Grid: **4 context lengths × 5 depths = 20 cells**.  
⚠️ Each cell costs ~3–5 GPT-4o calls (retrieval + generation + optional DeepEval). Estimated: ~10 min.

In [ ]:
NIAH_NEEDLE = (
    "A landmark RCT (Smith et al., 2019) found that the intervention "
    "reduced all-cause mortality by 42% (95% CI 31–51%, p<0.001), "
    "making it the largest effect size reported in this indication to date."
)
NIAH_QUESTION = (
    "What was the largest effect size reported for all-cause mortality "
    "reduction in the systematic review corpus?"
)
NIAH_EXPECTED = (
    "Smith et al. (2019) reported a 42% reduction in all-cause mortality "
    "(95% CI 31–51%, p<0.001)."
)

_HAYSTACK_FILLER = (
    "Multiple systematic reviews have examined the efficacy of pharmacological "
    "and non-pharmacological interventions across diverse patient populations. "
    "Heterogeneity in study design, outcome measurement, and follow-up duration "
    "limits direct comparability of findings. Meta-analytic pooling was performed "
    "using random-effects models to account for between-study variance. "
    "Publication bias was assessed via funnel plots and Egger's test. "
    "Risk of bias was evaluated independently by two reviewers using the "
    "Cochrane Risk of Bias 2.0 tool. Subgroup analyses stratified by age, "
    "sex, and comorbidity burden revealed no significant interaction effects. "
    "Future high-quality trials with standardised outcome reporting are needed. "
)

faithfulness_metric = FaithfulnessMetric(model=EVAL_MODEL, threshold=0.7, include_reason=True)
niah_results: dict[str, Any] = {}
total_cells = len(NIAH_CONTEXT_LENGTHS) * len(NIAH_DEPTHS)
run_n = 0

for ctx_len in NIAH_CONTEXT_LENGTHS:
    for depth in NIAH_DEPTHS:
        run_n += 1
        cell_key = f"ctx{ctx_len}_d{int(depth * 100)}"
        print(f"\n[niah] {run_n}/{total_cells}  ctx={ctx_len}  depth={depth:.0%}")

        # 1. Build haystack with needle at depth
        target_chars = ctx_len * 4
        filler_reps = max(1, target_chars // len(_HAYSTACK_FILLER) + 2)
        base = (_HAYSTACK_FILLER * filler_reps)[:target_chars]
        insert_pos = int(len(base) * depth)
        snap = base.rfind(". ", 0, insert_pos)
        insert_pos = snap + 2 if snap != -1 else insert_pos
        haystack_text = base[:insert_pos] + " " + NIAH_NEEDLE + " " + base[insert_pos:]

        # 2. Chunk and temporarily add to ChromaDB
        hay_doc = Document(
            page_content=haystack_text,
            metadata={"source": "niah_haystack", "title": f"NIAH ctx={ctx_len} d={depth}",
                      "year": "2024", "niah_cell": cell_key},
        )
        hay_chunks = splitter.split_documents([hay_doc])
        doc_ids = vs.add_documents(hay_chunks)

        # 3. Query
        t0 = time.perf_counter()
        res = run_query(NIAH_QUESTION)
        latency = round(time.perf_counter() - t0, 3)

        # 4. Score
        needle_key = "42% (95% CI 31"
        retrieved = any(needle_key in chunk for chunk in res["retrieval_context"])
        in_answer = "42%" in res["answer"] and "Smith" in res["answer"]

        faithfulness_metric.measure(LLMTestCase(
            input=NIAH_QUESTION,
            actual_output=res["answer"],
            retrieval_context=res["retrieval_context"],
            expected_output=NIAH_EXPECTED,
        ))
        faith_score = faithfulness_metric.score

        niah_results[cell_key] = {
            "context_length": ctx_len, "depth": depth,
            "needle_retrieved": retrieved, "needle_in_answer": in_answer,
            "faithfulness": faith_score, "latency_s": latency,
            "answer_snippet": res["answer"][:300],
        }

        status = "✓ PASS" if (retrieved and in_answer) else "✗ FAIL"
        print(f"  {status}  retrieved={retrieved}  in_answer={in_answer}  "
              f"faith={faith_score:.3f}  latency={latency}s")

        # 5. Remove temporary chunks
        vs.delete(ids=doc_ids)

# ASCII heatmap
print("\n── NIAH Heatmap (✓=pass  ✗=fail) ──")
print("ctx\\depth  " + "  ".join(f"{int(d*100):>4}%" for d in NIAH_DEPTHS))
for ctx in NIAH_CONTEXT_LENGTHS:
    row = ["  ✓ " if (niah_results.get(f"ctx{ctx}_d{int(d*100)}", {}).get("needle_retrieved")
                     and niah_results.get(f"ctx{ctx}_d{int(d*100)}", {}).get("needle_in_answer"))
           else "  ✗ " for d in NIAH_DEPTHS]
    print(f"{ctx:>9}  " + "".join(row))

# Save
niah_out = project_root() / "reports" / "niah_results_standard_rag.json"
niah_out.parent.mkdir(parents=True, exist_ok=True)
niah_out.write_text(json.dumps(niah_results, indent=2, default=str))
print(f"\n[niah] Results saved → {niah_out}")

## Full query-set run + experiment log

Run the standardised 10-query eval set (`data/splits/eval_query_set.csv`) through the chain, save results, and write the run record to `experiments/`.

In [ ]:
query_df = pd.read_csv(data_dir() / "splits" / "eval_query_set.csv")
full_results = []
t0 = time.perf_counter()

for _, row in query_df.iterrows():
    res = run_query(row["question"])
    res["query_id"] = row.get("query_id", "")
    full_results.append(res)

elapsed = round(time.perf_counter() - t0, 1)

# Flatten sources to string for CSV storage
results_rows = [
    {
        "query_id":  r["query_id"],
        "question":  r["question"],
        "answer":    r["answer"],
        "n_sources": len(r["sources"]),
        "source_titles": " | ".join(s["title"] for s in r["sources"]),
    }
    for r in full_results
]
results_df = pd.DataFrame(results_rows)
out_path = data_dir() / "processed" / "standard_rag_results.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(out_path, index=False)
print(f"[run] {len(full_results)} queries in {elapsed}s → {out_path}")

# Log experiment record
run_id = generate_run_id("standard_rag")
log_experiment({
    "run_id":          run_id,
    "method":          "standard_rag",
    "embed_model":     EMBED_MODEL,
    "llm_model":       LLM_MODEL,
    "top_k":           TOP_K,
    "chunk_size":      CHUNK_SIZE,
    "chunk_overlap":   CHUNK_OVERLAP,
    "n_docs_ingested": len(docs),
    "n_chunks":        len(chunks),
    "n_queries":       len(full_results),
    "time_seconds":    elapsed,
    "deepeval_scores": summary,
    "niah_pass_rate":  round(
        sum(1 for v in niah_results.values() if v["needle_retrieved"] and v["needle_in_answer"])
        / len(niah_results), 3
    ) if niah_results else None,
})
print(f"[log] Run {run_id} recorded → experiments/")